# 181. Employees Earning More Than Their Managers

**Difficulty:** Easy &nbsp;|&nbsp; **Topics:** database, self-join
&nbsp;|&nbsp; [LeetCode](https://leetcode.com/problems/employees-earning-more-than-their-managers/)

```
Table: Employee
+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| id          | int     |
| name        | varchar |
| salary      | int     |
| managerId   | int     |
+-------------+---------+
id is the primary key. Each row indicates the ID of an employee, their name, salary,
and the ID of their manager.
```

Write a solution to find the employees who earn **more** than their managers.

Return the result table in **any order**. The result column must be called `Employee`.

---

### Example

```
Employee:
+----+-------+--------+-----------+
| id | name  | salary | managerId |
+----+-------+--------+-----------+
| 1  | Joe   | 70000  | 3         |
| 2  | Henry | 80000  | 4         |
| 3  | Sam   | 60000  | Null      |
| 4  | Max   | 90000  | Null      |
+----+-------+--------+-----------+

Output:
+----------+
| Employee |
+----------+
| Joe      |
+----------+
```

Joe earns 70000 and his manager Sam earns 60000, so Joe is reported. Henry earns 80000
but his manager Max earns 90000, so he is not.

---

One table, joined to **itself**. Once that idea lands - that a table can appear twice
in the same query wearing two different hats - a whole category of problem opens up.

## Before you write anything

**1.** The rows you want compare **two rows of the same table**: an employee, and the
row whose `id` equals that employee's `managerId`. SQL can only compare columns within
one row of its result - so before you can compare, you must first build a result where
the employee and the manager are **side by side in one row**. Say what operation does
that.

**2.** So you need `Employee` twice. What goes wrong if you write
`FROM Employee JOIN Employee ON ...` with no aliases? Try it. Then write the aliased
version - `e` for the employee, `m` for the manager - and say which alias goes on which
side of the `ON`. Getting `e.managerId = m.id` the wrong way round returns *managers who
earn more than their reports*, which is a perfectly sensible query and the wrong answer.

**3.** Two of the four rows have `managerId = NULL`. What does a `JOIN` do with them?
Work out whether you need an explicit `WHERE e.managerId IS NOT NULL`, or whether the
join has already handled it - and be able to say **why**, in terms of what `NULL = 3`
evaluates to. (It is not `true` and it is not `false`.)

**4.** "Earn more than" - `>` or `>=`? Construct the one-line dataset that tells the two
apart, then check the statement's wording again. Boundary cases in SQL are exactly the
boundary cases from #707's index table: a different problem, the same discipline.

**5.** The output column must be named `Employee`, not `name`. Write the `AS`. Then say
what the result would look like without it, and why a grader that ignored column names
would be a bad grader.

**6.** What should happen for an employee who is their **own** manager (`id = managerId`)?
The statement does not say, so work out what your query does - and decide whether that
is a bug or a reasonable reading. There is a test case for it below; predict the answer
before you run it.

## Two routes

**A - the self join** *(write this first)*

```sql
SELECT e.name AS Employee
FROM Employee e
JOIN Employee m ON e.managerId = m.id
WHERE e.salary > m.salary
```

Read it as two separate tables that happen to share a name: `e` is "the employee",
`m` is "their manager". The `JOIN` is doing the real work - it pairs each employee with
exactly the row that is their manager, and silently drops the ones with no manager,
because `NULL = m.id` is never true. That is question 3's answer, and it means the
`IS NOT NULL` filter you were about to write is already free.

**B - a correlated subquery**

```sql
SELECT e.name AS Employee
FROM Employee e
WHERE e.salary > (SELECT m.salary FROM Employee m WHERE m.id = e.managerId)
```

Reads closer to the English sentence, and it is worth writing once to see the shape.
The comparison `70000 > NULL` is `NULL`, not `true`, so employees with no manager are
excluded here too - by a completely different mechanism, and it is worth being able to
say which mechanism is which.

Historically B was slower, because a naive engine runs the subquery once per row.
Modern planners usually rewrite it into A. Do not guess about performance - `EXPLAIN
QUERY PLAN` will tell you, and there is an exercise for it below.

> **`NULL` is not a value - it is "unknown", and it never compares true to anything.**
> That single rule silently drops the two managers here, and it is the same rule that
> will silently return *zero rows* in #183. Learn it in the problem where it helps you,
> so you recognise it in the problem where it ruins you.

In [ ]:
SOLUTION = '''
'''

### The test harness

Every notebook in this folder runs your SQL for real, against a fresh **SQLite**
database built from scratch for each test case. Nothing is mocked and nothing is
pattern-matched - if your query runs and returns the right rows, it passes.

`check(name, data, expected)` creates the tables, inserts that case's rows, executes
whatever string is in `SOLUTION`, and compares. It checks two things: the **rows**
(as a set - row order does not matter unless the problem says it does) and the
**column names**, because a query that returns the right numbers under the wrong
headings is not the answer the question asked for.

On failure it prints your rows next to the expected ones and names which rows are
missing and which should not be there.

`show(name, data, query)` is there for you: run *any* query against any dataset and
print it. Use it to look at intermediate results while you are working - especially
to run the deliberately-wrong version of your query and watch what it does.

> **SQLite here, MySQL on LeetCode.** They agree on everything these problems need -
> joins, `GROUP BY`/`HAVING`, subqueries, `LIMIT`/`OFFSET`, `COALESCE`, and window
> functions like `DENSE_RANK`. Where a problem needs something MySQL does differently,
> the notebook says so in the routes section. Write standard SQL and both will take it.

Run this cell; don't edit it.

In [ ]:
import sqlite3

SCHEMA = """CREATE TABLE Employee (id INTEGER, name TEXT, salary INTEGER, managerId INTEGER);"""

EXPECTED_COLUMNS = ['Employee']
ORDERED = False


def _norm(rows):
    return rows if ORDERED else sorted(rows, key=lambda r: tuple((v is None, str(v)) for v in r))


def check(name, data_sql, expected):
    """Build a fresh in-memory database, run SOLUTION against it, compare."""
    con = sqlite3.connect(":memory:")
    try:
        con.executescript(SCHEMA)
        if data_sql.strip():
            con.executescript(data_sql)
    except sqlite3.Error as e:
        print(f"FAIL {name}")
        print(f"       the harness could not build the tables: {e}")
        return False

    if not SOLUTION.strip():
        print(f"FAIL {name}")
        print("       SOLUTION is empty - write your query in the cell above")
        return False

    try:
        cur = con.execute(SOLUTION)
        got = [tuple(r) for r in cur.fetchall()]
        cols = [d[0] for d in cur.description] if cur.description else []
    except sqlite3.Error as e:
        print(f"FAIL {name}")
        print(f"       your query raised {type(e).__name__}: {e}")
        return False

    cols_ok = [c.lower() for c in cols] == [c.lower() for c in EXPECTED_COLUMNS]
    rows_ok = _norm(got) == _norm(expected)

    if cols_ok and rows_ok:
        print(f"OK   {name}")
        return True

    print(f"FAIL {name}")
    if not cols_ok:
        print(f"       column names  {cols}")
        print(f"       should be     {EXPECTED_COLUMNS}")
    if not rows_ok:
        missing = [r for r in expected if r not in got]
        extra = [r for r in got if r not in expected]
        print(f"       you returned {len(got)} row(s), expected {len(expected)}"
              + ("   (row order matters here)" if ORDERED else "   (row order does not matter)"))
        for r in got[:6]:
            print(f"         got       {r}")
        for r in expected[:6]:
            print(f"         expected  {r}")
        if missing:
            print(f"       rows you are MISSING: {missing[:4]}")
        if extra:
            print(f"       rows you should NOT have: {extra[:4]}")
    return False


def show(name, data_sql, query):
    """Run any query against a dataset and print it - for exploring, not for grading."""
    con = sqlite3.connect(":memory:")
    con.executescript(SCHEMA)
    if data_sql.strip():
        con.executescript(data_sql)
    cur = con.execute(query)
    cols = [d[0] for d in cur.description]
    rows = cur.fetchall()
    print(f"-- {name}")
    print("   " + " | ".join(str(c) for c in cols))
    for r in rows:
        print("   " + " | ".join("NULL" if v is None else str(v) for v in r))
    if not rows:
        print("   (no rows)")

In [ ]:
# tests
check("the LeetCode example", '''
INSERT INTO Employee VALUES (1,'Joe',70000,3), (2,'Henry',80000,4),
                            (3,'Sam',60000,NULL), (4,'Max',90000,NULL);
''', [('Joe',)])

check("nobody earns more than their manager", '''
INSERT INTO Employee VALUES (1,'Boss',100000,NULL), (2,'Worker',50000,1);
''', [])

check("everybody earns more than their manager", '''
INSERT INTO Employee VALUES (1,'Boss',10,NULL), (2,'A',20,1), (3,'B',30,1);
''', [('A',), ('B',)])

check("question 4: equal salaries do NOT count", '''
INSERT INTO Employee VALUES (1,'Boss',50000,NULL), (2,'Worker',50000,1);
''', [])

check("one more than the manager counts", '''
INSERT INTO Employee VALUES (1,'Boss',50000,NULL), (2,'Worker',50001,1);
''', [('Worker',)])

check("question 3: everyone is a manager of nobody", '''
INSERT INTO Employee VALUES (1,'A',10,NULL), (2,'B',20,NULL), (3,'C',30,NULL);
''', [])

check("a chain of managers, three deep", '''
INSERT INTO Employee VALUES (1,'Top',10,NULL), (2,'Mid',5,1), (3,'Low',100,2);
''', [('Low',)])

check("question 6: an employee who manages themselves", '''
INSERT INTO Employee VALUES (1,'Solo',5000,1);
''', [])

check("a manager id that does not exist", '''
INSERT INTO Employee VALUES (1,'Ghosted',9999,42);
''', [])

check("two employees with the same name are two rows", '''
INSERT INTO Employee VALUES (1,'Boss',10,NULL), (2,'Sam',20,1), (3,'Sam',30,1);
''', [('Sam',), ('Sam',)])

check("an empty table", '', [])

## After it passes

- **Swap the join condition** to `e.id = m.managerId` and run the tests. It will fail,
  and the rows it returns are *managers who out-earn a report* - a real query, answering
  a different question. Being able to look at a wrong result and say what question it
  *did* answer is a genuinely useful debugging skill.
- **Prove question 3.** Run `SELECT 70000 > NULL, 70000 = NULL, NULL = NULL` with
  `show`. Every answer is `NULL`, never `1` or `0`. Now say in one sentence why that is
  what drops the managers from your result without a `WHERE` clause.
- **Look at the plan.** Run `EXPLAIN QUERY PLAN` on route A and route B against the same
  data and compare. Then add `CREATE INDEX idx ON Employee(id)` and look again. This is
  the cheapest possible introduction to the fact that SQL says *what*, not *how*.
- **Make it a real report.** Report the employee, their manager's name, and the
  difference - and sort by the difference descending. That is the version somebody would
  actually ask you for, and it needs nothing you do not already have.
- Siblings: **#183 Customers Who Never Order** (the `NULL` rule again, this time as a
  trap), #182 Duplicate Emails, #184 Department Highest Salary (a join *plus* a group),
  #570 Managers with at Least 5 Direct Reports (this table, grouped the other way round).